# 🏠 Dubai Property Rental Price Prediction
### Complete ML Pipeline: Import → Clean → EDA → Feature Engineering → Scale → Model → Pickle
---

## 1. Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('dubai_properties.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Data Cleaning

In [ ]:
# Check missing values
print('Missing Values:')
print(df.isnull().sum())
print(f'\nTotal rows: {len(df)}')

In [ ]:
# Keep only relevant columns
keep_cols = ['Rent', 'Beds', 'Baths', 'Type', 'Area_in_sqft', 'Furnishing', 'Location', 'City']
df = df[keep_cols].copy()
df = df.dropna(subset=keep_cols)
print(f'After dropping NaN: {df.shape[0]} rows')

# Ensure numeric types
df['Rent'] = pd.to_numeric(df['Rent'], errors='coerce')
df['Beds'] = pd.to_numeric(df['Beds'], errors='coerce')
df['Baths'] = pd.to_numeric(df['Baths'], errors='coerce')
df['Area_in_sqft'] = pd.to_numeric(df['Area_in_sqft'], errors='coerce')
df = df.dropna()
print(f'After numeric cleanup: {df.shape[0]} rows')

In [ ]:
# Check duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'After removing duplicates: {df.shape[0]} rows')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Rent distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['Rent'], bins=50, color='#667eea', edgecolor='white')
axes[0].set_title('Rent Distribution')
axes[0].set_xlabel('Rent (AED/year)')
axes[1].boxplot(df['Rent'], vert=True)
axes[1].set_title('Rent Boxplot')
plt.tight_layout()
plt.show()

In [ ]:
# Value counts for categorical columns
for col in ['Type', 'Furnishing', 'City']:
    print(f'\n--- {col} ---')
    print(df[col].value_counts().head(10))

In [ ]:
# Rent by Type
fig, ax = plt.subplots(figsize=(10, 5))
df.groupby('Type')['Rent'].mean().sort_values(ascending=False).head(10).plot(kind='barh', color='#764ba2', ax=ax)
ax.set_title('Average Rent by Property Type')
ax.set_xlabel('Avg Rent (AED/year)')
plt.tight_layout()
plt.show()

In [ ]:
# Rent by City
fig, ax = plt.subplots(figsize=(10, 5))
df.groupby('City')['Rent'].mean().sort_values(ascending=False).head(10).plot(kind='barh', color='#00c9ff', ax=ax)
ax.set_title('Average Rent by City')
ax.set_xlabel('Avg Rent (AED/year)')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric columns
fig, ax = plt.subplots(figsize=(8, 6))
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Outlier Removal

In [ ]:
print(f'Before outlier removal: {df.shape[0]} rows')

# Rent outliers
q_low = df['Rent'].quantile(0.005)
q_high = df['Rent'].quantile(0.99)
df = df[(df['Rent'] >= q_low) & (df['Rent'] <= q_high)]
print(f'After rent outlier removal ({q_low:.0f} - {q_high:.0f}): {df.shape[0]} rows')

# Area outliers
q_area_high = df['Area_in_sqft'].quantile(0.99)
df = df[(df['Area_in_sqft'] > 0) & (df['Area_in_sqft'] <= q_area_high)]
print(f'After area outlier removal: {df.shape[0]} rows')

# Cap beds and baths
df = df[(df['Beds'] <= 12) & (df['Baths'] <= 15)]
print(f'After beds/baths cap: {df.shape[0]} rows')

## 5. Feature Engineering & Encoding

In [ ]:
# Clean Type - group rare types into 'Other'
type_counts = df['Type'].value_counts()
common_types = type_counts[type_counts >= 50].index.tolist()
df['Type_clean'] = df['Type'].apply(lambda x: x if x in common_types else 'Other')
print('Property Types:', df['Type_clean'].value_counts().to_dict())

In [ ]:
# Clean Furnishing
furnishing_map = {'Unfurnished': 'Unfurnished', 'Furnished': 'Furnished', 'Partly Furnished': 'Partly Furnished'}
df['Furnishing_clean'] = df['Furnishing'].map(furnishing_map).fillna('Unfurnished')
print('Furnishing:', df['Furnishing_clean'].value_counts().to_dict())

In [ ]:
# Clean Location - keep top locations
location_counts = df['Location'].value_counts()
top_locations = location_counts[location_counts >= 100].index.tolist()
df['Location_clean'] = df['Location'].apply(lambda x: x if x in top_locations else 'Other')
print(f'Locations kept: {len(top_locations)} + Other')

# Clean City
city_counts = df['City'].value_counts()
top_cities = city_counts[city_counts >= 50].index.tolist()
df['City_clean'] = df['City'].apply(lambda x: x if x in top_cities else 'Other')
print(f'Cities kept: {top_cities}')

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode all categorical features
type_encoder = LabelEncoder()
df['type_encoded'] = type_encoder.fit_transform(df['Type_clean'])
type_list = list(type_encoder.classes_)

furnishing_encoder = LabelEncoder()
df['furnishing_encoded'] = furnishing_encoder.fit_transform(df['Furnishing_clean'])
furnishing_list = list(furnishing_encoder.classes_)

location_encoder = LabelEncoder()
df['location_encoded'] = location_encoder.fit_transform(df['Location_clean'])
location_list = list(location_encoder.classes_)

city_encoder = LabelEncoder()
df['city_encoded'] = city_encoder.fit_transform(df['City_clean'])
city_list = list(city_encoder.classes_)

print(f'Type classes: {type_list}')
print(f'Furnishing classes: {furnishing_list}')
print(f'Location classes: {len(location_list)}')
print(f'City classes: {city_list}')

## 6. Feature Scaling & Train-Test Split

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

feature_columns = ['Beds', 'Baths', 'type_encoded', 'Area_in_sqft', 'furnishing_encoded', 'location_encoded', 'city_encoded']

X = df[feature_columns].copy()
Y = df['Rent'].copy()

print(f'Feature matrix: {X.shape}')
print(f'\nRent statistics (AED/year):')
print(Y.describe())

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=42)
print(f'\nTrain: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 7. Model Building & Comparison

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

models = {
    'LinearRegression': LinearRegression(),
    'KNN-5': KNeighborsRegressor(n_neighbors=5),
    'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, min_samples_leaf=3, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42, min_samples_leaf=5),
}

results = []
best_model = None
best_score = -999
best_name = ''

for name, model in models.items():
    model.fit(X_train, Y_train)
    train_r2 = model.score(X_train, Y_train)
    test_r2 = model.score(X_test, Y_test)
    Y_pred = model.predict(X_test)
    mae = mean_absolute_error(Y_test, Y_pred)
    rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
    cv = cross_val_score(model, X_scaled, Y, cv=5, scoring='r2')
    
    results.append({'Model': name, 'Train R²': train_r2, 'Test R²': test_r2, 'CV R²': cv.mean(), 'MAE': mae, 'RMSE': rmse})
    print(f'{name}: Train R²={train_r2:.4f}, Test R²={test_r2:.4f}, CV R²={cv.mean():.4f}, MAE={mae:,.0f}, RMSE={rmse:,.0f}')
    
    if test_r2 > best_score:
        best_score = test_r2
        best_model = model
        best_name = name

print(f'\n✅ Best Model: {best_name} (Test R² = {best_score:.4f})')

In [ ]:
# Visualize model comparison
results_df = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(results_df))
ax.bar([i-0.15 for i in x], results_df['Train R²'], 0.3, label='Train R²', color='#667eea')
ax.bar([i+0.15 for i in x], results_df['Test R²'], 0.3, label='Test R²', color='#764ba2')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=15)
ax.set_ylabel('R² Score')
ax.set_title('Model Comparison')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Save Model to Pickle

In [ ]:
import pickle

# Retrain best model on full data
best_model.fit(X_scaled, Y)
final_r2 = best_model.score(X_scaled, Y)
print(f'Final R² on full data: {final_r2:.4f}')

# Save artifacts
model_artifacts = {
    'model': best_model,
    'scaler': scaler,
    'feature_columns': feature_columns,
    'type_list': type_list,
    'furnishing_list': furnishing_list,
    'location_list': location_list,
    'city_list': city_list,
    'type_encoder': type_encoder,
    'furnishing_encoder': furnishing_encoder,
    'location_encoder': location_encoder,
    'city_encoder': city_encoder,
    'model_name': best_name,
    'r2_score': best_score,
    'num_samples': len(df),
}

with open('model.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print('\n✅ Model saved to model.pkl')
print(f'   Model: {best_name}')
print(f'   R² Score: {best_score:.4f}')
print(f'   Features: {feature_columns}')
print(f'   Types: {type_list}')
print(f'   Furnishing: {furnishing_list}')
print(f'   Locations: {len(location_list)} areas')
print(f'   Cities: {city_list}')

## 9. Sanity Check Predictions

In [ ]:
test_cases = [
    {'desc': '2-Bed Apt, 1200sqft, Unfurnished, Dubai Marina, Dubai', 'beds': 2, 'baths': 3, 'type': 'Apartment', 'area': 1200, 'furnishing': 'Unfurnished', 'location': 'Dubai Marina', 'city': 'Dubai'},
    {'desc': '3-Bed Villa, 3500sqft, Furnished, JVC, Dubai', 'beds': 3, 'baths': 4, 'type': 'Villa', 'area': 3500, 'furnishing': 'Furnished', 'location': 'Jumeirah Village Circle (JVC)', 'city': 'Dubai'},
    {'desc': '1-Bed Apt, 800sqft, Furnished, Al Reem Island, Abu Dhabi', 'beds': 1, 'baths': 2, 'type': 'Apartment', 'area': 800, 'furnishing': 'Furnished', 'location': 'Al Reem Island', 'city': 'Abu Dhabi'},
    {'desc': 'Studio, 450sqft, Unfurnished, Business Bay, Dubai', 'beds': 0, 'baths': 1, 'type': 'Apartment', 'area': 450, 'furnishing': 'Unfurnished', 'location': 'Business Bay', 'city': 'Dubai'},
]

for tc in test_cases:
    type_enc = type_list.index(tc['type']) if tc['type'] in type_list else type_list.index('Other')
    furn_enc = furnishing_list.index(tc['furnishing']) if tc['furnishing'] in furnishing_list else 0
    loc_enc = location_list.index(tc['location']) if tc['location'] in location_list else location_list.index('Other')
    city_enc = city_list.index(tc['city']) if tc['city'] in city_list else city_list.index('Other')
    features = [[tc['beds'], tc['baths'], type_enc, tc['area'], furn_enc, loc_enc, city_enc]]
    features_scaled = scaler.transform(features)
    pred = max(best_model.predict(features_scaled)[0], 0)
    print(f"{tc['desc']}")
    print(f"  → AED {pred:,.0f}/year  |  AED {pred/12:,.0f}/month\n")